# 🛒 매장 KPI 예측 모델 — PurchaseGridTransformer

**캡스톤 프로젝트 | 2026-05**

---

## 개요

시뮬레이터로 생성한 매장 구매 이벤트 데이터를 기반으로,  
**다음 날의 (구역 × 시간대) 셀별 구매량**을 예측하는 Transformer 모델입니다.

### 핵심 아이디어
- `7일치 구매 이력 grid [L, Zone×Hour, Feature]` → `내일 예측 [Zone×Hour]`
- **Spatial attention** (같은 시점 내 84개 토큰끼리) + **Temporal attention** (7일 시계열)
- 구역/시간대/요일을 분리 임베딩으로 학습

### 최종 성능 (Test set, 16일)
| 지표 | 값 |
|---|---|
| 구역별 MAPE | **4.0%** ✅ |
| 시간대별 MAPE | **5.4%** ✅ |
| 구역 순위 ρ (Spearman) | **1.000** ✅ |
| 시간대 순위 ρ | **0.972** ✅ |
| 셀 MAE (baseline 대비) | **−31%** 개선 |

---
## 0. 환경 설정

In [ ]:
# 필요 패키지 설치 (Colab 기본 환경에는 대부분 포함되어 있음)
!pip install -q pyyaml scipy

In [ ]:
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

### 데이터 업로드

아래 두 방법 중 하나를 선택하세요.

**방법 A — Google Drive 마운트** (권장, 반복 실행 시 편리)

In [ ]:
# 방법 A: Google Drive 마운트
# Drive에 data/ 폴더를 올려두었을 때 사용

# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/캡스톤/model/data')

# 방법 B: 직접 업로드 (한 번만 실행)
from google.colab import files
import io, os

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)

print("CSV 파일 3개를 업로드하세요:")
print("  - batch_purchases (1).csv")
print("  - batch_zone_kpi (1).csv")
print("  - batch_class_kpi (1).csv")
uploaded = files.upload()
for fname, data in uploaded.items():
    (DATA_DIR / fname).write_bytes(data)
    print(f"  저장: {DATA_DIR / fname}")

In [ ]:
# 설정값 (config.yaml 내용을 dict로 인라인)
CFG = {
    "data": {
        "purchases_csv":  str(DATA_DIR / "batch_purchases (1).csv"),
        "zone_kpi_csv":   str(DATA_DIR / "batch_zone_kpi (1).csv"),
        "window_size": 7,   # 입력 윈도우 (일)
        "horizon":     1,   # 예측 horizon (일)
        "train_days":  84,  # 1~84일 학습
        "val_days":    20,  # 85~104일 검증
    },
    "model": {
        "d_model":  96,
        "n_heads":   4,
        "n_layers":  3,
        "dropout":   0.1,
    },
    "train": {
        "batch_size":     32,
        "lr":             0.001,
        "weight_decay":   0.01,
        "epochs":         200,
        "patience":       30,
        "grad_clip":      1.0,
        "warmup_epochs":  5,
        "checkpoint_dir": "/content/checkpoints",
    },
}

Path(CFG['train']['checkpoint_dir']).mkdir(exist_ok=True)
print("설정 로드 완료")

---
## 1. 데이터 탐색 (EDA)

### 데이터 구조
| 파일 | 설명 | 행 수 |
|---|---|---|
| `batch_purchases` | 구매 이벤트 raw (메인 학습 입력) | ~84,406 |
| `batch_zone_kpi` | 구역 단위 집계 (sanity check) | 8 |
| `batch_class_kpi` | 고객군 단위 집계 (보조) | 4 |

In [ ]:
df = pd.read_csv(CFG['data']['purchases_csv'])
df['hour'] = df['시간대'].str.replace('시', '').astype(int)

print(f"행 수: {len(df):,}")
print(f"컬럼: {list(df.columns)}")
df.head()

In [ ]:
print("=== 카테고리 현황 ===")
print(f"구역 ({df['구역ID'].nunique()}개): {sorted(df['구역ID'].unique())}")
print(f"시간대 ({df['hour'].nunique()}개): {sorted(df['hour'].unique())}")
print(f"고객분류 ({df['고객분류'].nunique()}개): {sorted(df['고객분류'].unique())}")
print(f"일차 범위: {df['일차'].min()} ~ {df['일차'].max()} 일 (총 {df['일차'].nunique()}일)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 일별 구매량 추이
daily = df.groupby('일차').size()
axes[0].plot(daily.index, daily.values, linewidth=1)
axes[0].axvline(84, color='r', linestyle='--', label='train/val')
axes[0].axvline(104, color='orange', linestyle='--', label='val/test')
axes[0].set_title('Daily Purchase Count')
axes[0].set_xlabel('Day')
axes[0].legend()

# 시간대별 구매량
hourly = df.groupby('hour').size()
axes[1].bar(hourly.index, hourly.values)
axes[1].set_title('Purchase by Hour')
axes[1].set_xlabel('Hour')

# 구역별 구매량
zone = df.groupby('구역명').size().sort_values(ascending=True)
axes[2].barh(zone.index, zone.values)
axes[2].set_title('Purchase by Zone')

plt.tight_layout()
plt.show()

---
## 2. Dataset — PurchaseGridDataset

### 전처리 파이프라인

```
batch_purchases CSV
       ↓
(일차, 구역, 시간대) 기준 집계 → grid [D, Z, T, F]
       ↓
정규화 (활성 셀 기준 mean/std, count는 log1p 후 정규화)
       ↓
슬라이딩 윈도우 → (x [L, N, F], y [H, N], dow [L])
```

**입력 피처 8개 (F=8)**
| idx | 피처 | 정규화 방식 |
|---|---|---|
| 0 | 구매 이벤트 수 | log1p → z-score |
| 1 | 체류시간 평균 | z-score |
| 2 | 가격 평균 | z-score |
| 3 | 고객 수 (unique) | log1p → z-score |
| 4~7 | 고객분류 비중 (4종) | 그대로 [0,1] |

In [ ]:
class PurchaseGridDataset(Dataset):
    """
    batch_purchases CSV → (일차, 구역, 시간대) 집계 grid → 슬라이딩 윈도우

    입력 텐서: [window_size, Z*T, F]  (정규화된 피처)
    타겟 텐서: [horizon, Z*T]          (log1p(구매 이벤트 수))

    카테고리 매핑은 항상 전체 CSV 기준 → train/val/test 인덱스 일관성 보장.
    정규화 통계(mean/std)는 train 샘플에서 한 번 fit 후 stats 인자로 공유.
    """

    N_FEATURES = 8  # 구매수, 체류시간, 가격, 고객수, 고객군 비중 4채널

    def __init__(self, purchases_path: str, window_size: int = 7, horizon: int = 1,
                 day_range: tuple = None, stats: dict = None):
        df_all = pd.read_csv(purchases_path)
        df_all['hour'] = df_all['시간대'].str.replace('시', '').astype(int)

        # 카테고리 매핑 (전체 CSV 기준)
        self.zones   = sorted(df_all['구역ID'].unique())
        self.hours   = sorted(df_all['hour'].unique())
        self.classes = sorted(df_all['고객분류'].unique())
        self.zone2idx = {z: i for i, z in enumerate(self.zones)}
        self.hour2idx = {h: i for i, h in enumerate(self.hours)}
        self.cls2idx  = {c: i for i, c in enumerate(self.classes)}
        self.n_zones  = len(self.zones)
        self.n_hours  = len(self.hours)
        self.n_tokens = self.n_zones * self.n_hours

        # day_range 필터
        df = df_all if day_range is None else df_all[df_all['일차'].between(day_range[0], day_range[1])]
        days_present = sorted(df['일차'].unique())
        n_days = len(days_present)
        self.day2seq = {d: i for i, d in enumerate(days_present)}
        self.seq2day = {i: d for d, i in self.day2seq.items()}

        # 원본 grid 빌드 (정규화 전)
        self.grid_raw = self._build_grid(df, n_days)  # [D, Z, T, F]

        # 정규화 통계 fit/transform
        if stats is None:
            self.stats = self._fit_stats(self.grid_raw)
        else:
            self.stats = stats

        self.grid       = self._normalize(self.grid_raw, self.stats)     # 입력용 (정규화)
        self.target_raw = self.grid_raw[..., 0]                          # 구매 이벤트 수
        self.target     = np.log1p(self.target_raw).astype(np.float32)   # log1p 타겟

        self.window_size = window_size
        self.horizon     = horizon
        max_start        = n_days - window_size - horizon + 1
        self.indices     = list(range(max(0, max_start)))

    def _build_grid(self, df: pd.DataFrame, n_days: int) -> np.ndarray:
        Z, T = self.n_zones, self.n_hours
        F    = self.N_FEATURES
        grid = np.zeros((n_days, Z, T, F), dtype=np.float32)

        for (day, zone, hour), grp in df.groupby(['일차', '구역ID', 'hour']):
            if day not in self.day2seq:
                continue
            d, zi, ti = self.day2seq[day], self.zone2idx[zone], self.hour2idx[hour]

            grid[d, zi, ti, 0] = len(grp)
            grid[d, zi, ti, 1] = grp['체류시간'].mean()
            grid[d, zi, ti, 2] = grp['가격'].mean()
            grid[d, zi, ti, 3] = grp['에이전트ID'].nunique()
            total = len(grp)
            for cls, grp_cls in grp.groupby('고객분류'):
                ci = self.cls2idx.get(cls)
                if ci is not None:
                    grid[d, zi, ti, 4 + ci] = len(grp_cls) / total

        return grid

    @staticmethod
    def _fit_stats(grid: np.ndarray) -> dict:
        """count류 (0,3): log1p 후 mean/std. 연속값 (1,2): raw mean/std. 비중 (4~7): 그대로."""
        F    = grid.shape[-1]
        flat = grid.reshape(-1, F)
        nz_mask = flat[:, 0] > 0   # 활성 셀 (구매가 있는 셀만)
        active  = flat[nz_mask]

        means       = np.zeros(F, dtype=np.float32)
        stds        = np.ones(F,  dtype=np.float32)
        log_indices  = {0, 3}
        skip_indices = {4, 5, 6, 7}  # 비중은 이미 [0,1]

        for i in range(F):
            if i in skip_indices:
                continue
            col = active[:, i]
            if i in log_indices:
                col = np.log1p(col)
            means[i] = col.mean() if len(col) else 0.0
            stds[i]  = col.std()  + 1e-6 if len(col) else 1.0

        return {'means': means, 'stds': stds,
                'log_indices': list(log_indices), 'skip_indices': list(skip_indices)}

    @staticmethod
    def _normalize(grid: np.ndarray, stats: dict) -> np.ndarray:
        out     = grid.copy()
        means   = stats['means']
        stds    = stats['stds']
        log_idx  = set(stats['log_indices'])
        skip_idx = set(stats['skip_indices'])
        F = grid.shape[-1]
        for i in range(F):
            if i in skip_idx:
                continue
            if i in log_idx:
                out[..., i] = np.log1p(out[..., i])
            out[..., i] = (out[..., i] - means[i]) / stds[i]
        return out.astype(np.float32)

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        start = self.indices[idx]
        x = self.grid[start: start + self.window_size]                         # [L, Z, T, F]
        y = self.target[start + self.window_size:
                        start + self.window_size + self.horizon]                # [H, Z, T]

        days = [self.seq2day[start + i] for i in range(self.window_size)]
        dow  = np.array([(d - 1) % 7 for d in days], dtype=np.int64)           # [L]

        L, Z, T, F = x.shape
        x = x.reshape(L, Z * T, F)
        y = y.reshape(self.horizon, Z * T)

        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(dow)


# 데이터셋 생성 확인
dc = CFG['data']
W, H         = dc['window_size'], dc['horizon']
train_end    = dc['train_days']
val_end      = train_end + dc['val_days']

train_ds = PurchaseGridDataset(dc['purchases_csv'], W, H, day_range=(1, train_end))
val_ds   = PurchaseGridDataset(dc['purchases_csv'], W, H,
                                day_range=(train_end + 1, val_end), stats=train_ds.stats)

print(f"구역: {train_ds.n_zones}개, 시간대: {train_ds.n_hours}개, 토큰: {train_ds.n_tokens}개")
print(f"Train 샘플: {len(train_ds)}, Val 샘플: {len(val_ds)}")

x0, y0, dow0 = train_ds[0]
print(f"\n입력 shape: {x0.shape}  (L={W}, N={train_ds.n_tokens}, F=8)")
print(f"타겟 shape: {y0.shape}  (H={H}, N={train_ds.n_tokens})")
print(f"DoW shape:  {dow0.shape}")

---
## 3. 모델 아키텍처 — PurchaseGridTransformer

```
입력: [B, L=7, N=84, F=8]
        └── L: window (일)
        └── N: 7 zones × 12 hours = 84 tokens
        └── F: 8 피처

임베딩 (모두 d=96):
  zone_emb [7, d]   ─┐
  hour_emb [12, d]  ─┤→ token_emb (zone + hour)
  pos_emb  [L, d]   ─┘→ 일차 위치
  dow_emb  [7, d]   ──→ 요일

SpaceTimeBlock × 3:
  ① Spatial Attention  [B*L, N, d]  → 구역×시간대 공간 관계
  ② Temporal Attention [B*N, L, d]  → 7일 시계열 패턴
  ③ FFN (d → 4d → d) + GELU

Head: LayerNorm → Linear(d, horizon=1)
출력: [B, H=1, N=84]  (log1p 공간)
```

In [ ]:
class SpaceTimeBlock(nn.Module):
    """
    Spatial attention (한 시점 내 Z*T 토큰끼리) +
    Temporal attention (같은 토큰의 L 시점끼리) +
    FFN
    """

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.spatial_attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.temporal_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
        )
        self.norm_s = nn.LayerNorm(d_model)
        self.norm_t = nn.LayerNorm(d_model)
        self.norm_f = nn.LayerNorm(d_model)
        self.drop   = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, N, d]
        B, L, N, d = x.shape

        # 1. Spatial attention: 같은 시점 내 N개 토큰끼리
        xs = x.reshape(B * L, N, d)
        xs2, _ = self.spatial_attn(xs, xs, xs)
        x = self.norm_s(x + self.drop(xs2.reshape(B, L, N, d)))

        # 2. Temporal attention: 같은 토큰의 L 시점끼리
        xt = x.permute(0, 2, 1, 3).reshape(B * N, L, d)
        xt2, _ = self.temporal_attn(xt, xt, xt)
        x = self.norm_t(x + self.drop(xt2.reshape(B, N, L, d).permute(0, 2, 1, 3)))

        # 3. FFN
        x = self.norm_f(x + self.drop(self.ffn(x)))
        return x


class PurchaseGridTransformer(nn.Module):
    """
    구역×시간대 grid 시계열 → 다음 horizon일의 셀별 log1p(구매수) 예측

    입력:  x [B, L, N, F], dow [B, L]
    출력: [B, horizon, N]
    """

    def __init__(self, n_zones: int, n_hours: int, n_features: int,
                 d_model: int = 96, n_heads: int = 4, n_layers: int = 3,
                 dropout: float = 0.1, horizon: int = 1, max_window: int = 64):
        super().__init__()
        self.n_zones = n_zones
        self.n_hours = n_hours

        self.input_proj = nn.Linear(n_features, d_model)
        self.zone_emb   = nn.Embedding(n_zones,     d_model)
        self.hour_emb   = nn.Embedding(n_hours,     d_model)
        self.dow_emb    = nn.Embedding(7,           d_model)
        self.pos_emb    = nn.Embedding(max_window,  d_model)

        self.blocks = nn.ModuleList([
            SpaceTimeBlock(d_model, n_heads, dropout) for _ in range(n_layers)
        ])
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, horizon),
        )

    def forward(self, x: torch.Tensor, dow: torch.Tensor) -> torch.Tensor:
        B, L, N, _ = x.shape
        Z, T = self.n_zones, self.n_hours

        h = self.input_proj(x)  # [B, L, N, d]

        # zone × hour 분리 임베딩
        zone_ids  = torch.arange(Z, device=x.device).repeat_interleave(T)  # [N]
        hour_ids  = torch.arange(T, device=x.device).repeat(Z)             # [N]
        token_emb = self.zone_emb(zone_ids) + self.hour_emb(hour_ids)      # [N, d]
        h = h + token_emb.view(1, 1, N, -1)

        # 일차 위치 임베딩
        pos_ids = torch.arange(L, device=x.device).view(1, L, 1)           # [1, L, 1]
        h = h + self.pos_emb(pos_ids)

        # 요일 임베딩
        dow_e = self.dow_emb(dow).unsqueeze(2)                              # [B, L, 1, d]
        h = h + dow_e

        for block in self.blocks:
            h = block(h)

        out = self.head(h[:, -1, :, :])  # [B, N, horizon]
        return out.permute(0, 2, 1)       # [B, horizon, N]


# 모델 생성
mc = CFG['model']
n_features = x0.shape[2]

model = PurchaseGridTransformer(
    n_zones    = train_ds.n_zones,
    n_hours    = train_ds.n_hours,
    n_features = n_features,
    d_model    = mc['d_model'],
    n_heads    = mc['n_heads'],
    n_layers   = mc['n_layers'],
    dropout    = mc['dropout'],
    horizon    = H,
    max_window = max(W, 64),
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"모델 파라미터 수: {total_params:,}")
print(model)

---
## 4. 학습

### 학습 설정 요약
| 항목 | 값 |
|---|---|
| Optimizer | AdamW (lr=0.001, wd=0.01) |
| Loss | SmoothL1 (β=0.5) — 이상치 robust |
| LR Schedule | warmup 5 epoch → cosine decay |
| Early stopping | patience=30 |
| 타겟 공간 | log1p(구매수) |

> **왜 SmoothL1?** 구매 이벤트 수는 0이 많고 outlier가 존재합니다.
> MSE는 큰 오차에 과도하게 페널티를 부여하므로 SmoothL1이 더 안정적입니다.

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

tc = CFG['train']
train_loader = DataLoader(train_ds, batch_size=tc['batch_size'], shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=tc['batch_size'])

optimizer = torch.optim.AdamW(model.parameters(), lr=tc['lr'], weight_decay=tc['weight_decay'])
criterion = nn.SmoothL1Loss(beta=0.5)

epochs  = tc['epochs']
warmup  = tc['warmup_epochs']

def lr_lambda(epoch):
    if epoch < warmup:
        return (epoch + 1) / warmup
    progress = (epoch - warmup) / max(1, epochs - warmup)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

ckpt_dir  = Path(tc['checkpoint_dir'])
ckpt_dir.mkdir(exist_ok=True)

best_val   = float('inf')
no_improve = 0
patience   = tc['patience']
grad_clip  = tc['grad_clip']

train_losses, val_losses = [], []

print(f"Device: {device}  |  zones={train_ds.n_zones}  hours={train_ds.n_hours}  "
      f"tokens={train_ds.n_tokens}  features={n_features}")
print(f"Train samples={len(train_ds)}  Val samples={len(val_ds)}\n")

for epoch in range(1, epochs + 1):
    # --- Train ---
    model.train()
    train_loss = 0.0
    for x, y, dow in train_loader:
        x, y, dow = x.to(device), y.to(device), dow.to(device)
        pred = model(x, dow)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y, dow in val_loader:
            x, y, dow = x.to(device), y.to(device), dow.to(device)
            val_loss += criterion(model(x, dow), y).item() * x.size(0)
    val_loss /= max(len(val_ds), 1)

    scheduler.step()
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    improved = val_loss < best_val - 1e-5

    if epoch % 5 == 0 or improved:
        marker = ' *' if improved else ''
        print(f"Epoch {epoch:3d} | lr={scheduler.get_last_lr()[0]:.5f} | "
              f"train={train_loss:.4f} | val={val_loss:.4f}{marker}")

    if improved:
        best_val   = val_loss
        no_improve = 0
        torch.save({
            'epoch':      epoch,
            'model':      model.state_dict(),
            'val_loss':   float(val_loss),
            'zone2idx':   train_ds.zone2idx,
            'hour2idx':   train_ds.hour2idx,
            'cls2idx':    train_ds.cls2idx,
            'stats':      train_ds.stats,
            'n_zones':    train_ds.n_zones,
            'n_hours':    train_ds.n_hours,
            'n_features': n_features,
            'cfg_model':  mc,
            'cfg_data':   dc,
        }, ckpt_dir / 'best.pt')
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

print(f"\nBest val loss: {best_val:.4f}  →  {ckpt_dir / 'best.pt'}")

In [ ]:
# 학습 곡선
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('SmoothL1 Loss')
plt.title('Training Curve')
plt.legend()
plt.tight_layout()
plt.show()

---
## 5. 평가

### 평가 지표
- **셀 MAE**: baseline과의 개선폭 비교용
- **구역별 MAPE + Spearman ρ**: 구역 집계 구매량 정확도 및 순위 일치도
- **시간대별 MAPE + Spearman ρ**: 시간대 집계 구매량 정확도 및 순위 일치도
- **Baseline**: persistence (어제 = 오늘 예측)

In [ ]:
# best checkpoint 로드
ckpt = torch.load(ckpt_dir / 'best.pt', map_location=device, weights_only=False)

test_start = train_end + dc['val_days'] + 1
test_ds    = PurchaseGridDataset(dc['purchases_csv'], W, H,
                                  day_range=(test_start, 9999), stats=ckpt['stats'])
test_loader = DataLoader(test_ds, batch_size=64)

mc_ckpt = ckpt['cfg_model']
model_eval = PurchaseGridTransformer(
    n_zones    = ckpt['n_zones'],
    n_hours    = ckpt['n_hours'],
    n_features = ckpt['n_features'],
    d_model    = mc_ckpt['d_model'],
    n_heads    = mc_ckpt['n_heads'],
    n_layers   = mc_ckpt['n_layers'],
    dropout    = mc_ckpt['dropout'],
    horizon    = H,
    max_window = max(W, 64),
).to(device)
model_eval.load_state_dict(ckpt['model'])
model_eval.eval()

preds_log, trues_log = [], []
with torch.no_grad():
    for x, y, dow in test_loader:
        preds_log.append(model_eval(x.to(device), dow.to(device)).cpu().numpy())
        trues_log.append(y.numpy())
preds_log = np.concatenate(preds_log)
trues_log = np.concatenate(trues_log)

# log1p 역변환 → 원래 count 공간
preds = np.expm1(preds_log).clip(min=0)
trues = np.expm1(trues_log)

mae = np.abs(preds - trues).mean()
print(f"[셀 단위] MAE={mae:.4f}  (test samples={len(preds)})")

# === 구역/시간대 집계 지표 ===
Z = ckpt['n_zones']
T = ckpt['n_hours']
p = preds.reshape(-1, H, Z, T)
t = trues.reshape(-1, H, Z, T)

zone_pred = p.sum(axis=3).mean(axis=(0, 1))
zone_true = t.sum(axis=3).mean(axis=(0, 1))
hour_pred = p.sum(axis=2).mean(axis=(0, 1))
hour_true = t.sum(axis=2).mean(axis=(0, 1))

zone_mape = (np.abs(zone_pred - zone_true) / (zone_true + 1e-6)).mean() * 100
hour_mape = (np.abs(hour_pred - hour_true) / (hour_true + 1e-6)).mean() * 100
z_rho     = spearmanr(zone_pred, zone_true).statistic
t_rho     = spearmanr(hour_pred, hour_true).statistic

print(f"[구역별]   MAPE={zone_mape:.1f}%  순위 ρ={z_rho:.3f}")
print(f"[시간대별] MAPE={hour_mape:.1f}%  순위 ρ={t_rho:.3f}")

# === Baseline: persistence ===
p_last, t_y = [], []
grid = test_ds.target_raw
for s in test_ds.indices:
    p_last.append(grid[s + W - 1])
    t_y.append(grid[s + W])
p_last     = np.stack(p_last)
t_y        = np.stack(t_y)
base_mae   = np.abs(p_last - t_y).mean()
base_z_rho = spearmanr(p_last.sum(axis=2).mean(axis=0), t_y.sum(axis=2).mean(axis=0)).statistic

print(f"\n[baseline] persistence  cell MAE={base_mae:.4f}  zone ρ={base_z_rho:.3f}")
print(f"  → 모델이 baseline 대비 cell MAE {(base_mae - mae) / base_mae * 100:+.1f}% 개선")

---
## 6. 결과 시각화

In [ ]:
# 구역별 예측 vs 실측
idx2zone = {v: k for k, v in ckpt['zone2idx'].items()}
zone_labels = [idx2zone[i] for i in range(Z)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(Z)
width = 0.35
axes[0].bar(x_pos - width/2, zone_pred, width, label='Predicted', alpha=0.8)
axes[0].bar(x_pos + width/2, zone_true, width, label='Actual',    alpha=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(zone_labels, rotation=30, ha='right')
axes[0].set_title(f'Zone-level Purchase (MAPE={zone_mape:.1f}%, rho={z_rho:.3f})')
axes[0].set_ylabel('Avg Daily Count')
axes[0].legend()

# 시간대별 예측 vs 실측
idx2hour  = {v: k for k, v in ckpt['hour2idx'].items()}
hour_labels = [f"{idx2hour[i]}시" for i in range(T)]

axes[1].plot(range(T), hour_pred, 'o-', label='Predicted')
axes[1].plot(range(T), hour_true, 's--', label='Actual')
axes[1].set_xticks(range(T))
axes[1].set_xticklabels(hour_labels, rotation=45)
axes[1].set_title(f'Hourly Purchase (MAPE={hour_mape:.1f}%, rho={t_rho:.3f})')
axes[1].set_ylabel('Avg Daily Count')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 구역별 구매 비중 비교 (예측 vs 실측 vs KPI 집계)
zone_kpi = pd.read_csv(dc['zone_kpi_csv'])
df_raw   = pd.read_csv(dc['purchases_csv'])
id2name  = df_raw.drop_duplicates('구역ID').set_index('구역ID')['구역명'].to_dict()

pred_pct = zone_pred / zone_pred.sum() * 100
true_pct = zone_true / zone_true.sum() * 100

kpi_pct = []
zone_names = []
for zi in range(Z):
    zone_id   = idx2zone.get(zi, str(zi))
    zone_name = id2name.get(zone_id, zone_id)
    zone_names.append(zone_name)
    row = zone_kpi[zone_kpi['구역'] == zone_name]
    kpi_pct.append(row['구매고객비중(%)'].values[0] if len(row) else float('nan'))

print(f"{'구역':10s}  {'예측(%)':>8s}  {'실측(%)':>8s}  {'KPI(%)':>8s}")
for i, name in enumerate(zone_names):
    print(f"{name:10s}  {pred_pct[i]:8.1f}  {true_pct[i]:8.1f}  {kpi_pct[i]:8.1f}")

# 구매 비중 pie
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie(pred_pct, labels=zone_names, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Predicted')
axes[1].pie(true_pct, labels=zone_names, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Actual (Test)')
plt.suptitle('Zone Purchase Share (%)')
plt.tight_layout()
plt.show()

In [ ]:
# 구역×시간대 grid 히트맵 (예측 vs 실측)
pred_grid = p.mean(axis=(0, 1))   # [Z, T]
true_grid = t.mean(axis=(0, 1))   # [Z, T]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
vmax = max(pred_grid.max(), true_grid.max())

for ax, grid_data, title in zip(axes,
                                  [pred_grid, true_grid],
                                  ['Predicted', 'Actual']):
    im = ax.imshow(grid_data, aspect='auto', vmin=0, vmax=vmax, cmap='YlOrRd')
    ax.set_yticks(range(Z))
    ax.set_yticklabels(zone_names)
    ax.set_xticks(range(T))
    ax.set_xticklabels(hour_labels, rotation=45)
    ax.set_title(f'{title} — Zone×Hour Heatmap')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

---
## 7. 결과 요약

### v1 → v2 주요 개선 내역

| 개선 사항 | 효과 |
|---|---|
| 입력 정규화 추가 | 가격(1200~32000)이 다른 피처 압도하는 문제 해결 |
| 타겟 log1p 변환 | count data long-tail 분포 안정화 |
| zone/hour 분리 임베딩 + DoW 임베딩 | 공간·시간 구조 명시적 반영 |
| SmoothL1 + Cosine LR + Early stopping | 안정적 수렴 |

### 최종 성능 (v1 → v2)

| 지표 | v1 | v2 | 변화 |
|---|---|---|---|
| 셀 MAE | 3.58 | **2.45** | −31.6% |
| 구역별 MAPE | — | **4.0%** | ✅ |
| 시간대별 MAPE | — | **5.4%** | ✅ |
| 구역 순위 ρ | 0.500 | **1.000** | 7/7 완벽 |
| 시간대 순위 ρ | −0.462 | **0.972** | 음수→거의 완벽 |

### 향후 개선 방향
1. **방문 로그 추가** → 구매수 예측에서 **구매전환율** 예측으로 전환
2. **Multi-step horizon** → 1일 예측 → 7일 예측 확장
3. **실제 데이터 교체** → dummy 대신 실제 시뮬레이터 출력으로 재검증
4. **외부 피처** → 공휴일, 이벤트 여부 등 외생 변수 추가